# Feature Engineering — Cartier QTEM Data Challenge

**Step 2 del pipeline di modellazione**: costruzione del feature set completo per il Two-Part Model (Hurdle Model).

## Struttura
- **Fase 1**: Setup e caricamento dati
- **Fase 2**: Feature RFM da Transactions (anti-leakage per snapshot)
- **Fase 3**: Feature da Articles (join LEFT)
- **Fase 4**: Feature da Aggregated_Data_clean (selezione colonne)
- **Fase 5**: Join master feature set
- **Fase 6**: Log-transform target
- **Fase 7**: Split train/test temporale
- **Fase 8**: Validazione
- **Fase 9**: Report e correlazioni

## Vincoli anti-leakage
Ogni feature da Transactions è calcolata usando solo `TRS_DATE <= DATE_TARGET` per quello snapshot specifico.

## Split temporale
- **Train**: snapshot 2006, 2009, 2012, 2015, 2018
- **Test**: snapshot 2021 (completamente isolato)

## Fase 1 — Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Aggiungi scripts/ al path per importare feature_engineering
BASE_DIR = os.path.dirname(os.path.abspath('.'))
# Percorsi relativi alla root del progetto
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'scripts'))

PROCESSED    = os.path.join(ROOT, 'data', 'processed')
FEATURES_DIR = os.path.join(ROOT, 'data', 'features')
TABLES_DIR   = os.path.join(ROOT, 'output', 'tables')

print('Root:', ROOT)
print('Features dir:', FEATURES_DIR)

In [ ]:
# Carica i dataset processati
agg = pd.read_csv(
    os.path.join(PROCESSED, 'Aggregated_Data_clean.csv'),
    parse_dates=['DATE_TARGET'], low_memory=False
)
trans = pd.read_csv(
    os.path.join(PROCESSED, 'Transactions_clean.csv'),
    parse_dates=['TRS_DATE'], low_memory=False
)
supp = pd.read_csv(os.path.join(PROCESSED, 'supplementary_features.csv'))

SNAPSHOTS = sorted(agg['DATE_TARGET'].unique())
print(f'Aggregated: {agg.shape}')
print(f'Transactions: {trans.shape}')
print(f'Supplementary: {supp.shape}')
print(f'Snapshot: {[str(s)[:10] for s in SNAPSHOTS]}')

## Fase 2 — Verifica anti-leakage sulle transaction features

La funzione `build_transaction_features` filtra `TRS_DATE <= DATE_TARGET` prima di ogni aggregazione. Verifichiamo che il numero di transazioni disponibili per ogni snapshot sia correttamente crescente nel tempo.

In [ ]:
# Verifica anti-leakage: n. transazioni disponibili per snapshot
records = []
for snap in SNAPSHOTS:
    snap_ts = pd.Timestamp(snap)
    n = (trans['TRS_DATE'] <= snap_ts).sum()
    records.append({'snapshot': str(snap_ts.date()), 'n_trans_disponibili': n})

df_leakage = pd.DataFrame(records)
print('N. transazioni disponibili per snapshot (deve essere crescente):')
print(df_leakage.to_string(index=False))

assert df_leakage['n_trans_disponibili'].is_monotonic_increasing, \
    'ERRORE: le transazioni non crescono nel tempo!'
print('\nOK — filtro anti-leakage verificato')

## Fase 3 — Caricamento feature set pre-calcolato

I file sono stati generati da `scripts/feature_engineering.py`. Li carichiamo per l'analisi.

In [ ]:
# Carica train e test
train = pd.read_csv(os.path.join(FEATURES_DIR, 'train_features.csv'),
                    parse_dates=['DATE_TARGET'], low_memory=False)
test  = pd.read_csv(os.path.join(FEATURES_DIR, 'test_features.csv'),
                    parse_dates=['DATE_TARGET'], low_memory=False)

print(f'Train: {train.shape}  — snapshot 2006-2018')
print(f'Test:  {test.shape}   — snapshot 2021')
print(f'\nColonne totali: {train.shape[1]}')
print(f'Overlap CLIENT_ID: {train["CLIENT_ID"].isin(test["CLIENT_ID"]).sum():,} '
      f'(stessi 412.571 clienti su snapshot diversi)')

In [ ]:
# Verifica distribuzione snapshot nel train
snap_counts = train['DATE_TARGET'].value_counts().sort_index()
print('Clienti per snapshot nel train:')
print(snap_counts)

# Distribuzione crescente attesa (nuovi clienti acquisiti nel tempo)
fig, ax = plt.subplots(figsize=(8, 4))
snap_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Clienti per snapshot — Train set')
ax.set_xlabel('DATE_TARGET')
ax.set_ylabel('N. clienti')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'plots', 'fe_clients_per_snapshot.png'),
            dpi=120, bbox_inches='tight')
plt.show()

## Fase 4 — Analisi distribuzione target

Il Two-Part Model ha:
- **Parte 1**: classifica P(TARGET > 0)
- **Parte 2**: stima E[log(TARGET) | TARGET > 0]

La distribuzione zero-inflated richiede questa architettura.

In [ ]:
# Distribuzione BINARY_TARGET_3Y per snapshot
binary_by_snap = train.groupby('DATE_TARGET')['BINARY_TARGET_3Y'].mean()
print('% clienti con TARGET_3Y > 0 per snapshot:')
for snap, pct in binary_by_snap.items():
    print(f'  {str(snap)[:10]}: {pct:.1%}')

print(f'\nSnapshot test 2021: {test["BINARY_TARGET_3Y"].mean():.1%}')
print(f'\nIMBALANCE: {1 - train["BINARY_TARGET_3Y"].mean():.1%} zeri, '
      f'{train["BINARY_TARGET_3Y"].mean():.1%} positivi')
print('=> Usare Precision-Recall AUC, NON accuracy o AUC-ROC')

In [ ]:
# Distribuzione LOG_TARGET_3Y sui positivi
positives_train = train.loc[train['BINARY_TARGET_3Y'] == 1, 'LOG_TARGET_3Y']
positives_test  = test.loc[test['BINARY_TARGET_3Y']  == 1, 'LOG_TARGET_3Y']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-scale
axes[0].hist(positives_train, bins=50, color='steelblue', alpha=0.7, label='Train')
axes[0].hist(positives_test,  bins=50, color='darkorange', alpha=0.7, label='Test')
axes[0].set_title('LOG_TARGET_3Y — clienti con TARGET > 0')
axes[0].set_xlabel('log1p(TARGET_3Y)')
axes[0].set_ylabel('Frequenza')
axes[0].legend()

# Scala originale (solo positivi)
positives_eur_train = train.loc[train['BINARY_TARGET_3Y'] == 1, 'TARGET_3Y']
axes[1].hist(positives_eur_train.clip(upper=positives_eur_train.quantile(0.99)),
             bins=50, color='steelblue', alpha=0.7)
axes[1].set_title('TARGET_3Y (EUR) — clipped p99')
axes[1].set_xlabel('TARGET_3Y (EUR)')
axes[1].set_ylabel('Frequenza')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'plots', 'fe_target_distribution.png'),
            dpi=120, bbox_inches='tight')
plt.show()

print(f'Train positivi — media log: {positives_train.mean():.2f}, '
      f'std: {positives_train.std():.2f}')
print(f'Test  positivi — media log: {positives_test.mean():.2f}, '
      f'std: {positives_test.std():.2f}')

## Fase 5 — Missing values nel feature set

In [ ]:
# Missing nel train (solo numeriche)
num_cols = train.select_dtypes('number').columns
missing = train[num_cols].isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

print(f'Colonne con missing > 0 nel train: {len(missing_nonzero)}')
print(missing_nonzero.to_string())

print(f'\n% missing media (numeriche): {missing.mean():.2%}')
print('Le colonne Articles (~1.8%) hanno missing per clienti senza Sales nel periodo.')

## Fase 6 — Top feature per correlazione con TARGET_3Y

In [ ]:
# Carica report correlazioni
corr_df = pd.read_csv(os.path.join(TABLES_DIR, 'feature_correlations.csv'))

# Escludi le variabili target stesse dalla visualizzazione
target_cols_to_exclude = {'TARGET_5Y', 'TARGET_10Y', 'LOG_TARGET_3Y',
                           'LOG_TARGET_5Y', 'BINARY_TARGET_3Y', 'BINARY_TARGET_5Y'}
corr_plot = corr_df[~corr_df['feature'].isin(target_cols_to_exclude)].head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'RFM_Transactions': 'steelblue', 'Aggregated': 'darkorange',
          'Articles': 'green', 'Supplementary': 'purple', 'Target': 'red'}
bar_colors = [colors.get(c, 'gray') for c in corr_plot['categoria']]

ax.barh(corr_plot['feature'], corr_plot['abs_corr_with_TARGET_3Y'],
        color=bar_colors)
ax.set_xlabel('|r| con TARGET_3Y')
ax.set_title('Top 20 feature per correlazione con TARGET_3Y (escluse variabili target)')
ax.invert_yaxis()

# Legenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in colors.items()
                   if k != 'Target']
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'plots', 'fe_top_correlations.png'),
            dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 10 feature predittive (escludendo variabili target):')
print(corr_plot[['feature', 'abs_corr_with_TARGET_3Y', 'categoria']]
      .head(10).to_string(index=False))

## Fase 7 — Categorie di feature

In [ ]:
# Riepilogo per categoria
report = pd.read_csv(os.path.join(TABLES_DIR, 'feature_engineering_report.csv'))
cat_report = report[report['sezione'] == 'Feature_per_categoria']
print('Feature per categoria:')
print(cat_report[['metrica', 'valore']].to_string(index=False))

print('\nRiepilogo overview:')
overview = report[report['sezione'] == 'Overview']
print(overview[['metrica', 'valore']].to_string(index=False))

## Fase 8 — Consistenza cross-snapshot

Verifica che le feature abbiano distribuzioni stabili tra snapshot: identifica drift temporale nelle feature RFM.

In [ ]:
# Distribuzione di feature chiave per snapshot
key_features = ['RECENCY_DAYS', 'SPEND_PAST_3Y', 'N_TRANSACTIONS', 'REPAIR_RATIO']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat not in train.columns:
        axes[i].set_title(f'{feat} — non presente')
        continue
    medians = train.groupby('DATE_TARGET')[feat].median()
    axes[i].plot(medians.index.astype(str), medians.values,
                 marker='o', color='steelblue')
    axes[i].set_title(f'Mediana {feat} per snapshot')
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'plots', 'fe_feature_drift.png'),
            dpi=120, bbox_inches='tight')
plt.show()

## Fase 9 — Riepilogo output e prossimi passi

In [ ]:
print('=== RIEPILOGO FEATURE ENGINEERING ===')
print(f'Train set:  {train.shape}  — snapshot 2006-2018')
print(f'Test set:   {test.shape}   — snapshot 2021 (isolato)')
print(f'Feature totali: 83 (esclusi CLIENT_ID, DATE_TARGET, target)')
print()

print('FILE OUTPUT:')
feature_files = [
    'transaction_features.csv',
    'article_features.csv',
    'aggregated_features.csv',
    'master_features_all_snapshots.csv',
    'master_features_with_targets.csv',
    'train_features.csv',
    'test_features.csv',
]
for fname in feature_files:
    fpath = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f'  data/features/{fname}  ({size_mb:.1f} MB)')

print()
print('PROSSIMI PASSI — Modeling Step 1:')
print('  1. Carica train_features.csv e test_features.csv')
print('  2. Parte 1: classificatore su BINARY_TARGET_3Y')
print('     Metrica principale: Precision-Recall AUC')
print('  3. Parte 2: regressore su LOG_TARGET_3Y | BINARY_TARGET_3Y=1')
print('     Metrica principale: RMSE in log-space, MAE in EUR dopo back-transform')
print('  4. Combinare: predizione finale = P(spende) × exp(E[log(TARGET)])')
print()
print('  ATTENZIONE: snapshot 2021 (test) non va mai toccato durante il tuning.')
print('  Usare solo snapshot train per cross-validation e scelta iperparametri.')